In [2]:
import numpy as np
import pandas as pd
import os
import glob

# =======================
# Funciones del algoritmo ID3
# =======================

# Función para calcular la entropía de un conjunto de datos
def entropy(target_col):
    elements, counts = np.unique(target_col, return_counts=True)
    return np.sum([(-count / np.sum(counts)) * np.log2(count / np.sum(counts)) for count in counts])

# Función para calcular la ganancia de información de un atributo
def InfoGain(data, split_attribute_name, target_name):
    total_entropy = entropy(data[target_name])
    vals, counts = np.unique(data[split_attribute_name], return_counts=True)
    weighted_entropy = np.sum([
        (counts[i] / np.sum(counts)) * entropy(data[data[split_attribute_name] == vals[i]][target_name])
        for i in range(len(vals))
    ])
    return total_entropy - weighted_entropy

# Función para predecir una clase para una nueva fila utilizando el árbol ID3
def predict(row, tree):
    while isinstance(tree, dict):
        root_node = next(iter(tree))
        node_value = row.get(root_node, None)
        if node_value in tree[root_node]:
            tree = tree[root_node][node_value]
        else:
            # Si el valor no se encuentra en el árbol, se retorna la clase más común
            return most_common_class(tree[root_node])
    return tree

# Función para obtener la clase más común de un nodo (o subárbol)
def most_common_class(node):
    if isinstance(node, dict):
        leaf_values = []
        def get_leaf_values(subnode):
            if isinstance(subnode, dict):
                for value in subnode.values():
                    get_leaf_values(value)
            else:
                leaf_values.append(subnode)
        get_leaf_values(node)
        return max(set(leaf_values), key=leaf_values.count)
    return node

# Algoritmo ID3 recursivo
def ID3(data, originaldata, features, target_attribute_name, parent_node_class=None):
    # Caso base: Si no hay datos, se retorna la clase del nodo padre
    if len(data) == 0:
        return parent_node_class
    # Si todos los ejemplos pertenecen a la misma clase, se retorna esa clase
    elif len(np.unique(data[target_attribute_name])) <= 1:
        return data[target_attribute_name].iloc[0]
    # Si no quedan atributos para separar, se retorna la clase más frecuente en el nodo
    elif len(features) == 0:
        return np.unique(data[target_attribute_name])[np.argmax(np.unique(data[target_attribute_name], return_counts=True)[1])]
    else:
        # Se calcula la ganancia de información para cada atributo
        item_values = [InfoGain(data, feature, target_attribute_name) for feature in features]
        # Se obtiene el atributo con la mayor ganancia
        best_feature_index = np.argmax(item_values)
        best_feature = features[best_feature_index]
        tree = {best_feature: {}}
        # Se eliminan del listado de atributos a evaluar al elegir el mejor atributo
        features = [i for i in features if i != best_feature]
        # Se crea un subárbol para cada valor único del mejor atributo
        for value in np.unique(data[best_feature]):
            sub_data = data[data[best_feature] == value]
            subtree = ID3(
                sub_data, data, features, target_attribute_name,
                np.unique(data[target_attribute_name])[np.argmax(np.unique(data[target_attribute_name], return_counts=True)[1])]
            )
            tree[best_feature][value] = subtree
        return tree

# =======================
# Funciones para validación cruzada simple
# =======================

# Función que divide aleatoriamente los datos en 'folds' particiones
def cross_validation_split(data, folds):
    return np.array_split(data.sample(frac=1, random_state=42), folds)

# Función para realizar la validación cruzada (en este ejemplo se usa la división simple, no estratificada)
def cross_validate(data, folds, target_attribute_name, stratified=False):
    # En este ejemplo usaremos la división simple
    splits = cross_validation_split(data, folds)
    accuracies = []
    for i in range(len(splits)):
        test_set = splits[i]
        # Se combinan todas las particiones excepto la de prueba
        train_set = pd.concat(splits[:i] + splits[i+1:], ignore_index=True)
        features = [col for col in data.columns if col != target_attribute_name]
        # Se construye el árbol con los datos de entrenamiento
        tree = ID3(train_set, train_set, features, target_attribute_name)
        # Se predice para cada fila del conjunto de prueba
        predictions = test_set.apply(lambda row: predict(row, tree), axis=1)
        accuracy = (predictions == test_set[target_attribute_name]).mean()
        accuracies.append(accuracy)
    return accuracies

# =======================
# Procesamiento de múltiples bases de datos (archivos CSV)
# =======================

# Ruta de la carpeta que contiene los archivos CSV (por ejemplo, bases discretizadas)
carpeta_bd = r"C:\Users\Carlo\Desktop\IA\MDLP\base de datos discretizadas con mdlp R"

# Buscar todos los archivos CSV en la carpeta
archivos_csv = glob.glob(os.path.join(carpeta_bd, "*.csv"))

resultados = []

for archivo in archivos_csv:
    try:
        data = pd.read_csv(archivo)
        # Si existe la columna 'Sequence_Name', se elimina
        if 'Sequence_Name' in data.columns:
            data = data.drop('Sequence_Name', axis=1)
        # Verificar que la columna objetivo 'class' exista
        if 'class' not in data.columns:
            print(f"El archivo {os.path.basename(archivo)} no contiene la columna 'class'. Se omite.")
            continue

        # Se realiza la validación cruzada simple (10 particiones)
        accuracies = cross_validate(data, 10, "class", stratified=False)
        mean_acc = np.mean(accuracies)
        std_acc = np.std(accuracies)
        resultados.append({
            "Dataset": os.path.basename(archivo),
            "Mean Accuracy": mean_acc,
            "Std Deviation": std_acc
        })
        print(f"Procesado {os.path.basename(archivo)}: Precisión media = {mean_acc:.4f}, Desviación = {std_acc:.4f}")
    except Exception as e:
        print(f"Error al procesar {os.path.basename(archivo)}: {e}")

# Crear un DataFrame con los resultados
df_resultados = pd.DataFrame(resultados)
print("\nResultados finales:")
print(df_resultados)

c:\Users\Carlo\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Procesado dry_bean_mdlpR.csv: Precisión media = 0.8904, Desviación = 0.0069


c:\Users\Carlo\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Procesado glass_mdlpR.csv: Precisión media = 0.7297, Desviación = 0.1080
Procesado iris_mdlpR.csv: Precisión media = 0.9600, Desviación = 0.0442


c:\Users\Carlo\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
c:\Users\Carlo\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


...

Procesado winequality-red_mdlpR.csv: Precisión media = 0.5741, Desviación = 0.0343


c:\Users\Carlo\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Procesado winequality-white_mdlpR.csv: Precisión media = 0.5674, Desviación = 0.0216


c:\Users\Carlo\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Procesado yeast_mdlpR.csv: Precisión media = 0.5958, Desviación = 0.0471

Resultados finales:
                        Dataset  Mean Accuracy  Std Deviation
0            dry_bean_mdlpR.csv       0.890383       0.006882
1               glass_mdlpR.csv       0.729654       0.108036
2                iris_mdlpR.csv       0.960000       0.044222
3  letter_recognition_mdlpR.csv       0.805750       0.010950
4                rice_mdlpR.csv       0.919423       0.010169
5               seeds_mdlpR.csv       0.923810       0.057143
6     winequality-red_mdlpR.csv       0.574127       0.034290
7   winequality-white_mdlpR.csv       0.567371       0.021567
8               yeast_mdlpR.csv       0.595787       0.047056


Resultados de mi mdlp:

Resultados finales:
                       Dataset  Mean Accuracy  Std Deviation
0            dry_bean_mdlp.csv       0.890677       0.007546
1               gamma_mdlp.csv       0.819138       0.005004
2               glass_mdlp.csv       0.650216       0.147091
3                iris_mdlp.csv       0.673333       0.113333
4  letter_recognition_mdlp.csv       0.806100       0.010111
5          rice_cameo_mdlp.csv       0.916535       0.011608
6               seeds_mdlp.csv       0.923810       0.053026
7     winequality-red_mdlp.csv       0.579123       0.039607
8   winequality-white_mdlp.csv       0.564305       0.022044
9               yeast_mdlp.csv       0.570787       0.025755

REsultados con mi CAIM:

Resultados finales:
               Dataset  Mean Accuracy  Std Deviation
0    dry-bean_caim.csv       0.895745       0.009127
1       gamma_caim.csv       0.803996       0.010488
2       glass_caim.csv       0.683983       0.111702
3        iris_caim.csv       0.946667       0.065320
4      letter_caim.csv       0.774200       0.012929
5        rice_caim.csv       0.927559       0.009405
6       seeds_caim.csv       0.900000       0.049716
7    wine-red_caim.csv       0.600401       0.036340
8  wine-white_caim.csv       0.586557       0.020159
9       yeast_caim.csv       0.506784       0.048191

Resultados finales con mdlp R:

Resultados finales:
                        Dataset  Mean Accuracy  Std Deviation
0            dry_bean_mdlpR.csv       0.890383       0.006882
1               glass_mdlpR.csv       0.729654       0.108036
2                iris_mdlpR.csv       0.960000       0.044222
3  letter_recognition_mdlpR.csv       0.805750       0.010950
4                rice_mdlpR.csv       0.919423       0.010169
5               seeds_mdlpR.csv       0.923810       0.057143
6     winequality-red_mdlpR.csv       0.574127       0.034290
7   winequality-white_mdlpR.csv       0.567371       0.021567
8               yeast_mdlpR.csv       0.595787       0.047056